In [ ]:
# ==================== 2. RESULT VISUALIZATION ====================
if BENCHMARK_RESULTS_PATH.exists():
    # 2.1 Benchmark Table
    print("--- Primary Benchmark Results ---")
    records = []
    for r in benchmark_data['ranked_results']:
        records.append({
            "Method": r['method'],
            "wF1 Mean": r['f1_weighted_mean'],
            "wF1 Std": r['f1_weighted_std'],
            "mF1 Mean": r['f1_macro_mean'],
            "Accuracy": r['accuracy_mean']
        })
    df_results = pd.DataFrame(records)
    from IPython.display import display
    display(df_results)
    
    # 2.2 Confusion Matrices
    print("\n

In [ ]:
# ==================== 3. DEMO INFERENCE ====================
# We will demonstrate live inference using the ECAPA-TDNN model on a sample WAV file.
sample_path = ROOT_DIR / "sample_visec.wav"

if not sample_path.exists():
    print(f"❌ Sample audio {sample_path.name} not found.")
else:
    print(f"Playing sample audio: {sample_path.name}")
    import IPython.display as ipd
    from IPython.display import display
    display(ipd.Audio(str(sample_path)))

    # Load ECAPA Model
    from ECAPA.predict_emotion import EmotionClassifier
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_labels = len(emotion_labels)
    
    ecapa_model = EmotionClassifier(num_labels).to(device)
    checkpoint_path = ROOT_DIR / "ECAPA" / "emotion_model" / "best_ecapa_model.pth"
    
    if checkpoint_path.exists():
        ecapa_model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
        ecapa_model.eval()
        
        # Load and preprocess audio
        audio, sr = librosa.load(str(sample_path), sr=16000)
        from transformers import AutoFeatureExtractor
        processor = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
        inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
        features = inputs.input_values.to(device)
        
        # Predict
        with torch.no_grad():
            outputs = ecapa_model(features)
            probs = torch.softmax(outputs, dim=1)
            pred_idx = torch.argmax(probs, dim=1).item()
            conf = probs[0, pred_idx].item()
            
        print(f"\n